# Pré-processamento de Dados da PNS 2019

## Objetivo
Este notebook tem como objetivo preparar os dados da Pesquisa Nacional de Saúde (PNS) 2019 para análise, com foco em:
- Selecionar e renomear colunas relevantes
- Recodificar variáveis categóricas
- Tratar valores ausentes e outliers
- Pré-processar dados para modelagem
- Analisar a distribuição da variável alvo (asma)

## 1-Configuração Inicial

Importação das bibliotecas necessárias para o pré-processamento:

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## 2-Carregamento dos Dados

Vamos carregar o dataset original `dt_pns2019.csv` que contém os dados da pesquisa.

In [2]:
data = pd.read_csv("dt_pns2019.csv")
print("OK")

OK


## 3-Seleção e Renomeação de Variáveis

Selecionamos 51 colunas relevantes para nossa análise e renomeamos para nomes mais descritivos.

**Justificativa**: Os nomes originais (como "Q074", "C006") são códigos difíceis de interpretar. Os novos nomes seguem o padrão:
- `num_` para variáveis numéricas
- `catl_` para variáveis categóricas nominais
- `ord_` para variáveis ordinais
- `cath_` para variáveis categóricas hierárquicas

In [3]:
# Selecionar as colunas desejadas
cols_to_select = [
    "Q074", "C006", "C009", "J007", "C008", "V0031",
    "A001", "A002010", "A005010", "A01501", "A016010",
    "A02201", "A02306", "A02305", "J001", "J003",
    "J012", "J05402", "J037", "E01401", "M011011",
    "M011021", "M011031", "M011061", "I00102",
    "J02901", "J05404", "J002", "Q078", "J00402",
    "J01002", "J014", "J021", "M011051", "M011071",
    "A009010", "B003", "I006", "I00401", "I00402",
    "I00403", "Q075", "Q076", "Q07601", "Q07704",
    "Q07708", "M01001", "B001", "Q07706", "Q07705",
    "Q07707"
]

# Criar um DataFrame apenas com as colunas selecionadas
data_selected = data[cols_to_select].copy()

# Remover linhas onde "Q074" é NaN (se necessário)
data_selected = data_selected[data_selected["Q074"].notna()]

# Lista dos novos nomes das colunas
var_names = [
    "num_asma", "catl_sexo", "catl_cor_raca", "catl_doenca_cronica", "num_idade", "catl_tipo_de_area",
    "catl_tipo_de_domicilio", "cath_material_parede", "cath_fonte_de_agua", "cath_destino_do_esgoto",
    "catl_destino_do_lixo", "catl_animais", "catl_cachoros", "catl_gatos", "ord_saude", "catl_atividade",
    "num_consultas", "catl_homeopatia", "catl_internacao", "cath_trabalho", "catl_quimicos", "catl_ruidos",
    "catl_sol", "catl_material_biologico", "catl_plano_saude", "catl_medicamentos", "catl_plantas_medicinais",
    "catl_deixar_atividades", "ord_limita_asma", "cath_principal_motivo", "cath_local_atendimento",
    "catl_servico_saude", "catl_recebeu_atendimento", "catl_lixo", "catl_poeira_mineral", "cath_estado_agua",
    "ord_agente_saude", "ord_nivel_plano", "catl_plano_consultas", "catl_plano_exames", "catl_plano_internacoes",
    "num_idade_diagnostico", "catl_crise_asma", "catl_remedio_asma", "ord_remedio_semana", "catl_aerossol_asma",
    "catl_cigarro", "catl_domicilio_cadastrado", "ord_remedio_sus", "ord_farmacia_popular", "catl_pagou_remedio"
]

# Verificar se o número de colunas corresponde ao número de nomes
if len(cols_to_select) == len(var_names):
    # Renomear as colunas usando um dicionário de mapeamento
    rename_dict = dict(zip(cols_to_select, var_names))
    data_selected = data_selected.rename(columns=rename_dict)
    
    print("Colunas renomeadas com sucesso!")
    print(data_selected.head())
else:
    print("Erro: O número de colunas selecionadas não corresponde ao número de nomes em var_names.")
    print(f"Colunas selecionadas: {len(cols_to_select)}, Nomes fornecidos: {len(var_names)}")

Colunas renomeadas com sucesso!
    num_asma  catl_sexo  catl_cor_raca  catl_doenca_cronica  num_idade  \
0        2.0        2.0            1.0                  1.0       55.0   
9        2.0        2.0            4.0                  2.0       19.0   
10       2.0        2.0            2.0                  2.0       45.0   
18       2.0        2.0            2.0                  2.0       58.0   
19       2.0        2.0            4.0                  2.0       28.0   

    catl_tipo_de_area  catl_tipo_de_domicilio  cath_material_parede  \
0                   1                     1.0                   1.0   
9                   1                     1.0                   1.0   
10                  1                     1.0                   1.0   
18                  1                     1.0                   1.0   
19                  1                     2.0                   1.0   

    cath_fonte_de_agua  cath_destino_do_esgoto  ...  num_idade_diagnostico  \
0                 

## 4-Recodificação de Variáveis

Aplicamos um dicionário de recodificação para transformar códigos numéricos em rótulos descritivos.

**Exemplo**: 
- `num_asma`: 2 → 0 (Não), 1 → 1 (Sim), 9 → None
- `catl_sexo`: 0 → "Masculino", 1 → "Feminino", 9 → None

**Benefício**: Melhora a interpretabilidade dos dados para análise.

In [4]:
# Dicionário de recodificação (simples, direto)
dict_recode = {
    "num_asma": {2: 0, 1: 1, 9: None},

    "catl_sexo": {1: "Masculino", 2: "Feminino", 9: None},

    "catl_cor_raca": {
        1: "Branca", 2: "Preta", 3: "Amarela", 4: "Parda", 5: "Indigena", 9: None,
        "5.0": "Indigena"   # caso sujo no CSV
    },

    # Doença crônica — pega "2.0" também
    "catl_doenca_cronica": {
        1: "Sim", 2: "Não", 9: None,
        "2": "Não", "2.0": "Não"
    },

    # Tipo de área — PNS: 1=Capital, 2=Resto de RM, 3=Resto da UF, 4=RIDE
    "catl_tipo_de_area": {
        1: "Capital", 2: "Resto de RM", 3: "Resto da UF", 4: "RIDE", 9: None,
    },

    "catl_tipo_de_domicilio": {1: "Casa", 2: "Apartamento", 3: "Cortiço", 9: None},

    "cath_material_parede": {
        1: "Alvenaria revestida", 2: "Alvenaria sem revestimento", 3: "Taipa",
        4: "Aparelhada", 5: "Madeira aproveitada", 6: "Outro", 9: None
    },

    "cath_fonte_de_agua": {
        1: "Rede geral", 2: "Poço profundo ou artesiano", 3: "Poço raso",
        4: "Fonte ou nascente", 5: "Água da chuva", 6: "Outra", 9: None
    },

    "cath_destino_do_esgoto": {
        1: "Rede geral", 2: "Fossa séptica ligada à rede",
        3: "Fossa séptica não ligada à rede", 4: "Fossa rudimentar",
        5: "Vala", 6: "Rio/lago", 7: "Outra", 9: None
    },

    "catl_destino_do_lixo": {
        1: "Coletado",
        2: "Coletado em caçamba de serviço de limpeza",
        3: "Queimado", 4: "Enterrado", 5: "Terreno baldio", 6: "Outro", 9: None,
        " Coletado em caçamba de serviço de limpeza": "Coletado em caçamba de serviço de limpeza"
    },

    "catl_animais": {1: "Sim", 2: "Não"},
    "ord_saude": {1: "Muito bom", 2: "Bom", 3: "Regular", 4: "Ruim", 5: "Muito ruim", 9: None},
    "catl_homeopatia": {1: "Sim", 2: "Não", 9: None},
    "catl_internacao": {1: "Sim", 2: "Não", 9: None},

    "cath_trabalho": {
        1: "Doméstico", 2: "Militar", 3: "Setor privado", 4: "Setor público",
        5: "Empregador", 6: "Autônomo", 7: "Trabalho não remunerado", 9: None
    },

    "catl_quimicos": {1: "Sim", 2: "Não", 9: None},
    "catl_ruidos": {1: "Sim", 2: "Não", 9: None},
    "catl_sol": {1: "Sim", 2: "Não", 9: None},
    "catl_material_biologico": {1: "Sim", 2: "Não", 9: None},
    "catl_plano_saude": {1: "Sim", 2: "Não", 9: None},
    "catl_medicamentos": {1: "Sim", 2: "Não", 9: None},
    "catl_plantas_medicinais": {1: "Sim", 2: "Não", 9: None},
    "catl_deixar_atividades": {1: "Sim", 2: "Não", 9: None},

    "ord_limita_asma": {
        1: "Não limita", 2: "Um pouco", 3: "Moderadamente",
        4: "Intensamente", 5: "Muito intensamente", 9: None
    },

    "cath_principal_motivo": {
        1: "Problemas nos ossos e articulações",
        2: "Dor de cabeça ou enxaqueca",
        3: "Problemas gineco-obstétricos",
        4: "Problema odontológico/Dor de dente",
        5: "Problemas respiratórios",
        6: "Problemas gastrointestinais",
        7: "Dengue, Chikungunya, Zika Vírus ou Febre amarela",
        8: "Problemas cardiovasculares",
        9: "Diabetes",
        10: "Câncer",
        11: "Problemas neurológicos",
        12: "Saúde mental",
        13: "Lesões ou fraturas por acidentes ou violência",
        14: "Outro problema de saúde",
        99: None
    },

    "cath_local_atendimento": {
        1: "Farmácia",
        2: "Unidade básica de saúde",
        3: "Policlínica pública/PAM/Centro de Especialidades",
        4: "UPA/Pronto atendimento público",
        5: "Ambulatório de hospital público",
        6: "Consultório particular/Clínica privada",
        7: "Pronto atendimento de hospital privado",
        8: "Atendimento domiciliar",
        9: "Outro serviço",
        99: None
    },

    "catl_servico_saude": {1: "Sim", 2: "Não", 9: None},
    "catl_recebeu_atendimento": {1: "Sim", 2: "Não", 9: None},
    "catl_lixo": {1: "Sim", 2: "Não", 9: None},
    "catl_poeira_mineral": {1: "Sim", 2: "Não", 9: None},

    "cath_estado_agua": {
        1: "Filtrada", 2: "Fervida", 3: "Tratada com hipoclorito de sódio (cloro)",
        4: "Tratada de outra forma no domicílio", 5: "Mineral industrializada",
        6: "Sem tratamento no domicílio", 9: None
    },

    "ord_agente_saude": {
        1: "Mensalmente", 2: "A cada 2 meses", 3: "De 2 a 4 vezes",
        4: "Uma vez", 5: "Nunca recebeu", 9: None
    },

    "ord_nivel_plano": {
        1: "Muito bom", 2: "Bom", 3: "Regular", 4: "Ruim",
        5: "Muito ruim", 6: "Nunca usou o plano de saúde", 9: None
    },

    "catl_plano_consultas": {1: "Sim", 2: "Não", 9: None},
    "catl_plano_exames": {1: "Sim", 2: "Não", 9: None},
    "catl_plano_internacoes": {1: "Sim", 2: "Não", 9: None},

    "num_idade_diagnostico": {99: None},

    "catl_crise_asma": {1: "Sim", 2: "Não", 9: None},
    "catl_remedio_asma": {1: "Sim", 2: "Não", 9: None},

    "ord_remedio_semana": {1: "Sim, todos", 2: "Sim, alguns", 3: "Não, nenhum", 9: None},

    "catl_aerossol_asma": {1: "Sim", 2: "Não", 9: None},
    "catl_cigarro": {1: "Sim", 2: "Não", 9: None},
    "catl_domicilio_cadastrado": {1: "Sim", 2: "Não", 3: "Não sabe", 9: None},

    "ord_remedio_sus": {1: "Sim, todos", 2: "Sim, alguns", 3: "Não, nenhum", 9: None},
    "ord_farmacia_popular": {1: "Sim, todos", 2: "Sim, alguns", 3: "Não, nenhum", 9: None},

    "catl_pagou_remedio": {1: "Sim", 2: "Não", 9: None}
}

# Aplicar a recodificação (simples)
data_recodificado = data_selected.replace(dict_recode)

# Visualizar
print(data_recodificado.head())


    num_asma catl_sexo catl_cor_raca catl_doenca_cronica  num_idade  \
0        0.0  Feminino        Branca                 Sim       55.0   
9        0.0  Feminino         Parda                 Não       19.0   
10       0.0  Feminino         Preta                 Não       45.0   
18       0.0  Feminino         Preta                 Não       58.0   
19       0.0  Feminino         Parda                 Não       28.0   

   catl_tipo_de_area catl_tipo_de_domicilio cath_material_parede  \
0            Capital                   Casa  Alvenaria revestida   
9            Capital                   Casa  Alvenaria revestida   
10           Capital                   Casa  Alvenaria revestida   
18           Capital                   Casa  Alvenaria revestida   
19           Capital            Apartamento  Alvenaria revestida   

   cath_fonte_de_agua cath_destino_do_esgoto  ... num_idade_diagnostico  \
0          Rede geral       Fossa rudimentar  ...                   NaN   
9           Po

## 5- Salvando os Dados Selecionados

**Nesta etapa, vamoss**:
1. Criar um novo DataFrame contendo apenas as colunas selecionadas
2. Salvar esse subconjunto em um novo arquivo CSV para uso posterior

**Detalhes**:
- Estamos selecionando apenas as 51 colunas definidas anteriormente em `cols_to_select`
- O arquivo será salvo com:
  - `index=False`: Não incluir o índice do DataFrame
  - `encoding="utf-8"`: Para preservar caracteres especiais e acentos
- O shape final (90846, 51) mostra que mantivemos todas as linhas originais com as 51 colunas selecionadas

In [5]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
# selecionar colunas
data = data[cols_to_select]

# salvar em csv
data_selected.to_csv(
    "dt_dados_pns.csv",
    index=False,  # Não salva o índice (mais limpo)
    encoding="utf-8"  # Garante acentos e caracteres especiais
)
print(f"Shape final: {data_selected.shape}")
print("OK")

Shape final: (90846, 51)
OK


## 6- Análise de Valores Ausentes (Antes do Tratamento)

**Objetivo desta análise**:
- Quantificar a presença de dados faltantes em cada coluna do dataset
- Identificar padrões de missingness que possam indicar problemas na coleta
- Planejar estratégias adequadas de imputação ou remoção

**Métricas importantes**:
- Contagem absoluta de NAs por coluna
- Percentual de missing em relação ao total de registros
- Padrão de distribuição dos missings (aleatório ou sistemático)

**O que observar**:
1. Colunas com mais de 30% de valores ausentes podem exigir:
   - Remoção da coluna (se pouco relevante)
   - Imputação especial (como categoria "desconhecido" para variáveis categóricas)
2. Colunas com menos de 5% geralmente permitem:
   - Imputação simples (média, moda ou mediana)
   - Exclusão pontual de registros

**Próximos passos**:
- Cruzar com análise de importância das variáveis
- Decidir estratégia por coluna baseada no impacto analítico

In [6]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
print("Missings antes (%):\n", (data_recodificado.isna().sum() / len(data_recodificado)) * 100)


Missings antes (%):
 num_asma                      0.000000
catl_sexo                     0.000000
catl_cor_raca                 0.011008
catl_doenca_cronica           0.000000
num_idade                     0.000000
catl_tipo_de_area             0.000000
catl_tipo_de_domicilio        0.000000
cath_material_parede          0.000000
cath_fonte_de_agua            0.000000
cath_destino_do_esgoto        1.491535
catl_destino_do_lixo          0.000000
catl_animais                  0.000000
catl_cachoros                40.663320
catl_gatos                   40.663320
ord_saude                     0.000000
catl_atividade               90.285758
num_consultas                21.242542
catl_homeopatia              93.780684
catl_internacao               0.000000
cath_trabalho                41.844440
catl_quimicos                41.844440
catl_ruidos                  41.844440
catl_sol                     41.844440
catl_material_biologico      41.844440
catl_plano_saude              0.000000
catl

## 7- Separação de Variáveis Numéricas e Categóricas

Nesta etapa, vamos classificar as colunas do dataset em dois grupos:

**1. Variáveis Numéricas** (`numeric_cols`):
- Colunas com tipos `int64` e `float64`
- Representam valores quantitativos (ex: idade, renda)

**2. Variáveis Categóricas** (`categorical_cols`):
- Colunas com tipos `object` e `category`
- Representam valores qualitativos (ex: sexo, cor da pele)

**Objetivo**:
- Preparar os dados para pré-processamento específico por tipo:
  - Numéricas: normalização, tratamento de outliers
  - Categóricas: one-hot encoding, label encoding

**Método**:
- Usamos `select_dtypes()` para filtrar colunas por tipo

In [7]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
# 1. Separar variáveis numéricas e categóricas
numeric_cols = data_recodificado.select_dtypes(include=['int64', 'float64']).columns
numeric_cols = numeric_cols.drop('num_asma')  # Remover a target
categorical_cols = data_recodificado.select_dtypes(include=['object', 'category']).columns

print(f"Variáveis numéricas: {len(numeric_cols)} colunas")
print(f"Variáveis categóricas: {len(categorical_cols)} colunas")

Variáveis numéricas: 6 colunas
Variáveis categóricas: 44 colunas


## 8- Tratamento de Outliers nas Variáveis Numéricas

**Objetivo**:
- Reduzir o impacto de valores extremos que podem distorcer análises e modelos
- Manter a consistência dos dados sem perder informações relevantes

**Estratégia Adotada**:
- Método: **Winsorization** (limitação dos valores extremos)
- Limites: 
  - `lower=quantil(0.01)` (remove os 1% menores valores)
  - `upper=quantil(0.99)` (remove os 1% maiores valores)
- Técnica: `clip()` para truncar os valores além dos limites

**Impacto**:
- Preserva 98% da distribuição original de cada variável
- Minimiza distorções mantendo a estrutura geral dos dados

**Atenção**:
- Para variáveis com distribuição muito assimétrica, outros métodos (como transformação log) podem ser mais adequados
- Em casos específicos, a remoção completa dos outliers pode ser considerada

In [8]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
# Tratamento simples de outliers
# Primeiro, vamos identificar as colunas numéricas
numeric_cols_temp = [col for col in data_recodificado.columns 
                     if col.startswith('num_') and col != 'num_asma']

for col in numeric_cols_temp:
    if col in data_recodificado.columns:
        data_recodificado[col] = data_recodificado[col].clip(
            lower=data_recodificado[col].quantile(0.01),
            upper=data_recodificado[col].quantile(0.99)
        )
        print(f"Outliers tratados em: {col}")

Outliers tratados em: num_idade
Outliers tratados em: num_consultas
Outliers tratados em: num_idade_diagnostico


## 9 — Divisão treino/teste e exportação em CSV
Nesta etapa, separamos `X` e `y`, dividimos em treino (80%) e teste (20%), salvamos os arquivos CSV (`X_train`, `X_test`, `y_train`, `y_test`) e também o dataset **pré-processado completo**. 
Incluímos ainda uma checagem rápida de valores ausentes em `X_train`.

In [9]:
# --- Divisão 80/20 e exportação ---
# GARANTIA: remover NaN no alvo antes de dividir
data_recodificado = data_recodificado.dropna(subset=['num_asma'])

alvo = "num_asma"
X = data_recodificado.drop(columns=[alvo])
y = data_recodificado[alvo]

from sklearn.model_selection import train_test_split

print("Dividindo dados em treino e teste (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}  y_test: {y_test.shape}")
print(f"Distribuição treino - 0: {(y_train==0).sum()} | 1: {(y_train==1).sum()}")
print(f"Distribuição teste  - 0: {(y_test==0).sum()} | 1: {(y_test==1).sum()}")

# Salvar dataset completo pré-processado (com alvo)
data_recodificado.to_csv("dados_preprocessados.csv", index=False)

# Salvar os conjuntos separados
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("\nArquivos gerados:")
print("- dados_preprocessados.csv")
print("- X_train.csv | X_test.csv | y_train.csv | y_test.csv")

# Checagem de missings no X_train
import pandas as pd
miss_rate = pd.DataFrame(X_train.isna().sum() / len(X_train))
miss_rate.columns = ["missing_rate"]
miss_rate.sort_values("missing_rate", ascending=False, inplace=True)
miss_rate.head(30)  # mostra as 30 piores se existirem

Dividindo dados em treino e teste (80/20)...
X_train: (72676, 50)  X_test: (18170, 50)
y_train: (72676,)  y_test: (18170,)
Distribuição treino - 0: 69102 | 1: 3574
Distribuição teste  - 0: 17277 | 1: 893

Arquivos gerados:
- dados_preprocessados.csv
- X_train.csv | X_test.csv | y_train.csv | y_test.csv


,missing_rate
ord_remedio_sus,0.992804
ord_farmacia_popular,0.991056
catl_pagou_remedio,0.991056
ord_remedio_semana,0.984355
catl_aerossol_asma,0.984355
catl_remedio_asma,0.981851
catl_recebeu_atendimento,0.970073
ord_limita_asma,0.950823
num_idade_diagnostico,0.950823
catl_crise_asma,0.950823
